# Treinamento com interface de alto nível

## Importação das bibliotecas

In [31]:
# http://pytorch.org/
from os.path import exists

import torch

In [32]:
import argparse
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.optim.lr_scheduler import StepLR

## Criação da rede

In [33]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.entrada = nn.Linear(21632, 1568)

        self.meio = nn.Linear(1568, 3136)

        self.meio2 = nn.Linear(3136, 1568)

        self.saida = nn.Linear(1568, 10)

    def forward(self, x):
        x = self.conv1(x)

        x = torch.flatten(x, 1)
        x = self.entrada(x)
        x = F.relu(x)
        x = self.meio(x)
        x = F.relu(x)
        x = self.meio2(x)
        x = F.sigmoid(x)
        x = self.saida(x)

        output = F.log_softmax(x, dim=1)
        return output

model = Net()
model

Net(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1))
  (entrada): Linear(in_features=21632, out_features=1568, bias=True)
  (meio): Linear(in_features=1568, out_features=3136, bias=True)
  (meio2): Linear(in_features=3136, out_features=1568, bias=True)
  (saida): Linear(in_features=1568, out_features=10, bias=True)
)

## Treinamento

### Criando o objeto de treinamento

In [34]:
def train(log_interval, dry_run, model, device, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % log_interval == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))
            if dry_run:
                break

In [35]:
def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item()  # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)

    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        100. * correct / len(test_loader.dataset)))

## Avaliação

In [36]:
use_cuda = torch.cuda.is_available()

torch.manual_seed(1111)

device = torch.device("cuda" if use_cuda else "cpu")

train_kwargs = {'batch_size': 64}
test_kwargs = {'batch_size': 1000}
if use_cuda:
    cuda_kwargs = {'num_workers': 1,
                    'pin_memory': True,
                    'shuffle': True}
    train_kwargs.update(cuda_kwargs)
    test_kwargs.update(cuda_kwargs)

transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
    ])
dataset1 = datasets.MNIST('../data', train=True, download=True,
                    transform=transform)
dataset2 = datasets.MNIST('../data', train=False,
                    transform=transform)
train_loader = torch.utils.data.DataLoader(dataset1,**train_kwargs)
test_loader = torch.utils.data.DataLoader(dataset2, **test_kwargs)

model = Net().to(device)
optimizer = optim.RMSprop(model.parameters(), lr=0.01)

epochs = 14
scheduler = StepLR(optimizer, step_size=1, gamma=0.7)

for epoch in range(1, epochs + 1):
    train(100, False, model, device, train_loader, optimizer, epoch)
    test(model, device, test_loader)
    scheduler.step()

torch.save(model.state_dict(), "mnist_cnn.pt")

Train Epoch: 1 [0/60000 (0%)]	Loss: 2.360823
Train Epoch: 1 [6400/60000 (11%)]	Loss: 6.101956
Train Epoch: 1 [12800/60000 (21%)]	Loss: 8.068712
Train Epoch: 1 [19200/60000 (32%)]	Loss: 8.288488
Train Epoch: 1 [25600/60000 (43%)]	Loss: 10.982988
Train Epoch: 1 [32000/60000 (53%)]	Loss: 8.468035
Train Epoch: 1 [38400/60000 (64%)]	Loss: 7.249021
Train Epoch: 1 [44800/60000 (75%)]	Loss: 7.979891
Train Epoch: 1 [51200/60000 (85%)]	Loss: 4.783261
Train Epoch: 1 [57600/60000 (96%)]	Loss: 11.048698

Test set: Average loss: 9.9343, Accuracy: 1375/10000 (14%)

Train Epoch: 2 [0/60000 (0%)]	Loss: 8.347651
Train Epoch: 2 [6400/60000 (11%)]	Loss: 3.954144
Train Epoch: 2 [12800/60000 (21%)]	Loss: 5.668139
Train Epoch: 2 [19200/60000 (32%)]	Loss: 4.221101
Train Epoch: 2 [25600/60000 (43%)]	Loss: 5.818844
Train Epoch: 2 [32000/60000 (53%)]	Loss: 5.121628
Train Epoch: 2 [38400/60000 (64%)]	Loss: 6.571703
Train Epoch: 2 [44800/60000 (75%)]	Loss: 4.255363
Train Epoch: 2 [51200/60000 (85%)]	Loss: 5.414252